# Final Deploy-Safe Model Training


## Objective

This notebook trains the final deployable version of the model using only features that are available at inference time.

The goals are to:

- build a leakage-free training pipeline,
- train deployable XGBoost models for `ball_land_x` and `ball_land_y`,
- preserve reproducibility and artifact tracking,
- and produce the final model artifacts used in the Hugging Face application.

## Deployment Constraint

Only features that are known before prediction can be used in the deployed model.

Therefore, target-derived features such as:

- `dx_to_ball`
- `dy_to_ball`
- `dist_to_ball`

are excluded from this training pipeline.

## Output

This notebook produces:

- deployable trained models,
- feature column definitions,
- validation predictions,
- and training metadata

saved under:

`models/final_deploy_xgboost/`

In [1]:
import sys
from pathlib import Path
import json
from datetime import datetime
import joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

sys.path.append(str(Path("..").resolve()))

from src.config import PROCESSED_DIR, MODELS_DIR

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

##  Load v3 Dataset

We begin by loading the v3 processed dataset, which contains the richest feature set built during experimentation.

From this dataset, we retain only deploy-safe features for final training.

In [2]:
input_path = PROCESSED_DIR / "baseline_train_v3.parquet"
df = pd.read_parquet(input_path)

print("Loaded v3 dataset shape:", df.shape)
display(df.head())

Loaded v3 dataset shape: (152305, 48)


,game_id,play_id,nfl_id,player_name,frame_id,x,y,s,a,dir,o,absolute_yardline_number,player_weight,player_height_inches,player_age,is_moving_right,player_position,player_side,player_role,ball_land_x,ball_land_y,num_frames_output,dist_to_ball,speed_x,speed_y,acc_vector,x_centered,y_centered,momentum,speed_times_dir,x_times_speed,speed_bin_code,dx_to_ball,dy_to_ball,acc_x,acc_y,dir_minus_o,momentum_x,momentum_y,dist_from_field_center,x_prev,y_prev,s_prev,a_prev,delta_x,delta_y,delta_s,delta_a
0,2023090700,101,44930,Josh Reynolds,1,41.03,12.17,0.00,0.00,156.35,80.97,42,196,75,28,1,WR,Offense,Targeted Receiver,63.259998,-0.22,21,25.449655,-0.000000,0.000000,0.00,-18.97,-14.48,0.00,0.0000,0.0000,0,22.229998,-12.39,-0.000000,0.000000,75.38,-0.000000,0.000000,23.864855,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,2023090700,101,44930,Josh Reynolds,2,41.03,12.17,0.00,0.00,119.09,82.26,42,196,75,28,1,WR,Offense,Targeted Receiver,63.259998,-0.22,21,25.449655,-0.000000,0.000000,0.00,-18.97,-14.48,0.00,0.0000,0.0000,0,22.229998,-12.39,-0.000000,0.000000,36.83,-0.000000,0.000000,23.864855,41.03,12.17,0.00,0.00,0.00,28.86,41.03,41.03
2,2023090700,101,44930,Josh Reynolds,3,41.05,12.18,0.02,0.47,65.03,83.33,42,196,75,28,1,WR,Offense,Targeted Receiver,63.259998,-0.22,21,25.437060,0.008443,0.018131,0.47,-18.95,-14.47,3.92,1.3006,0.8210,0,22.209998,-12.40,0.198408,0.426069,-18.30,1.654803,3.553593,23.842890,41.03,12.17,0.00,0.00,0.02,28.88,41.05,41.05
3,2023090700,101,44930,Josh Reynolds,4,41.07,12.20,0.18,1.54,56.06,84.29,42,196,75,28,1,WR,Offense,Targeted Receiver,63.259998,-0.22,21,25.429361,0.100498,0.149332,1.54,-18.93,-14.45,35.28,10.0908,7.3926,0,22.189998,-12.42,0.859820,1.277619,-28.23,19.697686,29.269089,23.814857,41.05,12.18,0.02,0.47,0.02,28.89,41.05,40.60
4,2023090700,101,44930,Josh Reynolds,5,41.11,12.22,0.57,3.09,59.41,88.21,42,196,75,28,1,WR,Offense,Targeted Receiver,63.259998,-0.22,21,25.404252,0.290068,0.490674,3.09,-18.89,-14.43,111.72,33.8637,23.4327,0,22.149998,-12.44,1.572474,2.659967,-28.80,56.853323,96.172023,23.770928,41.07,12.20,0.18,1.54,0.04,28.91,40.93,39.57


## Define Deploy-Safe Features

We define the final modeling feature set using only variables that are available before prediction time.

Target-derived spatial features are excluded to ensure the model is valid for real-world inference and deployment.

In [3]:
target_cols = ["ball_land_x", "ball_land_y"]

feature_cols = [
    # Baseline features
    "x",
    "y",
    "s",
    "a",
    "dir",
    "o",
    "absolute_yardline_number",
    "player_weight",
    "player_height_inches",
    "player_age",
    "is_moving_right",
    "player_position",
    "player_side",
    "player_role",

    # Deploy-safe engineered features
    "speed_x",
    "speed_y",
    "acc_x",
    "acc_y",
    "dir_minus_o",
    "momentum",
    "momentum_x",
    "momentum_y",
    "x_centered",
    "y_centered",
    "dist_from_field_center",
    "x_prev",
    "y_prev",
    "s_prev",
    "a_prev",
    "delta_x",
    "delta_y",
    "delta_s",
    "delta_a",
    "speed_bin_code",
]

categorical_cols = [
    "player_position",
    "player_side",
    "player_role",
]

X = df[feature_cols].copy()
y_x = df["ball_land_x"].copy()
y_y = df["ball_land_y"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape (ball_land_x):", y_x.shape)
print("Target shape (ball_land_y):", y_y.shape)
print("\nCategorical columns:", categorical_cols)

Feature matrix shape: (152305, 34)
Target shape (ball_land_x): (152305,)
Target shape (ball_land_y): (152305,)

Categorical columns: ['player_position', 'player_side', 'player_role']


## Missing Value Check

We verify that the final deploy-safe feature matrix is clean and ready for training.

In [5]:
missing_percent = X.isnull().mean().sort_values(ascending=False)
display(missing_percent[missing_percent > 0])

print("Missing target values:")
print("ball_land_x:", y_x.isnull().sum())
print("ball_land_y:", y_y.isnull().sum())

Series([], dtype: float64)

Missing target values:
ball_land_x: 0
ball_land_y: 0


## Train / Validation Split

We use the same grouped split by `game_id` to keep evaluation consistent with previous iterations.

In [6]:
unique_games = df["game_id"].dropna().unique()
unique_games = np.sort(unique_games)

rng = np.random.default_rng(42)
shuffled_games = rng.permutation(unique_games)

split_idx = int(0.8 * len(shuffled_games))
train_games = set(shuffled_games[:split_idx])
valid_games = set(shuffled_games[split_idx:])

train_mask = df["game_id"].isin(train_games)
valid_mask = df["game_id"].isin(valid_games)

print("Number of unique games:", len(unique_games))
print("Training games:", len(train_games))
print("Validation games:", len(valid_games))

Number of unique games: 32
Training games: 25
Validation games: 7


## Encode Categorical Variables

Categorical variables are one-hot encoded to prepare the data for XGBoost.

In [7]:
X_encoded = pd.get_dummies(X, columns=categorical_cols, dummy_na=True)

X_train = X_encoded.loc[train_mask].copy()
X_valid = X_encoded.loc[valid_mask].copy()

y_train_x = y_x.loc[train_mask].copy()
y_valid_x = y_x.loc[valid_mask].copy()

y_train_y = y_y.loc[train_mask].copy()
y_valid_y = y_y.loc[valid_mask].copy()

print("Encoded training shape:", X_train.shape)
print("Encoded validation shape:", X_valid.shape)
print("Training rows:", X_train.shape[0])
print("Validation rows:", X_valid.shape[0])

Encoded training shape: (121480, 52)
Encoded validation shape: (30825, 52)
Training rows: 121480
Validation rows: 30825


##  Naive Baseline Benchmark

We compute a mean-value benchmark to confirm that the trained models add predictive value.

In [8]:
baseline_pred_x = np.full_like(y_valid_x, y_train_x.mean(), dtype=float)
baseline_pred_y = np.full_like(y_valid_y, y_train_y.mean(), dtype=float)

baseline_metrics_x = {
    "MAE": mean_absolute_error(y_valid_x, baseline_pred_x),
    "RMSE": np.sqrt(mean_squared_error(y_valid_x, baseline_pred_x)),
    "R2": r2_score(y_valid_x, baseline_pred_x),
}

baseline_metrics_y = {
    "MAE": mean_absolute_error(y_valid_y, baseline_pred_y),
    "RMSE": np.sqrt(mean_squared_error(y_valid_y, baseline_pred_y)),
    "R2": r2_score(y_valid_y, baseline_pred_y),
}

baseline_metrics_df = pd.DataFrame({
    "ball_land_x_baseline": baseline_metrics_x,
    "ball_land_y_baseline": baseline_metrics_y,
})

display(baseline_metrics_df)

,ball_land_x_baseline,ball_land_y_baseline
MAE,20.976016,13.747809
RMSE,25.432954,15.854705
R2,-0.000529,-0.000052


## Model Selection

We use XGBoost as the final deployable model because it performs well on structured data and can effectively exploit the engineered motion and interaction features.

In [9]:
model_x = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_y = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

## Train Final Deployable Models

We fit two separate XGBoost regressors:

- one for `ball_land_x`
- one for `ball_land_y`

In [10]:
model_x.fit(X_train, y_train_x)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=400, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [11]:
model_y.fit(X_train, y_train_y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=400, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

## Generate Validation Predictions

We generate validation predictions while preserving the original validation indices for later evaluation and debugging.

In [12]:
pred_valid_x = model_x.predict(X_valid)
pred_valid_y = model_y.predict(X_valid)

pred_df = pd.DataFrame({
    "ball_land_x_true": y_valid_x.values,
    "ball_land_x_pred": pred_valid_x,
    "ball_land_y_true": y_valid_y.values,
    "ball_land_y_pred": pred_valid_y,
}, index=y_valid_x.index)

pred_df["error_x"] = pred_df["ball_land_x_pred"] - pred_df["ball_land_x_true"]
pred_df["error_y"] = pred_df["ball_land_y_pred"] - pred_df["ball_land_y_true"]

display(pred_df.head())

,ball_land_x_true,ball_land_x_pred,ball_land_y_true,ball_land_y_pred,error_x,error_y
6371,55.57,49.599598,27.799999,27.309286,-5.970402,-0.490713
6372,55.57,49.351562,27.799999,26.757483,-6.218437,-1.042517
6373,55.57,48.448483,27.799999,26.270155,-7.121517,-1.529844
6374,55.57,48.495541,27.799999,25.672918,-7.074459,-2.127081
6375,55.57,48.572060,27.799999,25.561207,-6.997940,-2.238792


## Validation Performance

We compute the main regression metrics for both targets.

In [13]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

metrics_x = regression_metrics(y_valid_x, pred_valid_x)
metrics_y = regression_metrics(y_valid_y, pred_valid_y)

metrics_df = pd.DataFrame({
    "ball_land_x_model": metrics_x,
    "ball_land_y_model": metrics_y,
})

display(metrics_df)

,ball_land_x_model,ball_land_y_model
MAE,6.375578,7.668929
RMSE,8.537832,9.744014
R2,0.887246,0.622270


##  Model vs Baseline Comparison

We compare the deployable XGBoost models against the naïve benchmark to verify that the model adds meaningful predictive value.

In [14]:
comparison_df = pd.concat([baseline_metrics_df, metrics_df], axis=1)
display(comparison_df)

,ball_land_x_baseline,ball_land_y_baseline,ball_land_x_model,ball_land_y_model
MAE,20.976016,13.747809,6.375578,7.668929
RMSE,25.432954,15.854705,8.537832,9.744014
R2,-0.000529,-0.000052,0.887246,0.622270


##  Feature Importance

We inspect which deploy-safe features contribute most to the final model.

In [15]:
importance_x = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model_x.feature_importances_
}).sort_values("importance", ascending=False)

importance_y = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model_y.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 10 features for ball_land_x:")
display(importance_x.head(10))

print("Top 10 features for ball_land_y:")
display(importance_y.head(10))

Top 10 features for ball_land_x:


,feature,importance
0,x,0.271493
19,x_centered,0.261151
29,delta_a,0.224320
10,is_moving_right,0.084769
21,dist_from_field_center,0.014742
28,delta_s,0.014207
33,player_position_DT,0.010187
6,absolute_yardline_number,0.008722
4,dir,0.006922
44,player_position_WR,0.006487


Top 10 features for ball_land_y:


,feature,importance
20,y_centered,0.364892
1,y,0.150420
17,momentum_x,0.059647
13,acc_x,0.042929
30,speed_bin_code,0.027093
11,speed_x,0.025162
49,player_role_Defensive Coverage,0.020953
46,player_side_Defense,0.017470
10,is_moving_right,0.014008
47,player_side_Offense,0.013434


## Save Deployable Artifacts

We save the final models and metadata used by the Hugging Face application.

In [16]:
run_dir = MODELS_DIR / "final_deploy_xgboost"
run_dir.mkdir(parents=True, exist_ok=True)

model_x_path = run_dir / "model_ball_land_x.pkl"
model_y_path = run_dir / "model_ball_land_y.pkl"
features_path = run_dir / "feature_columns.json"
metadata_path = run_dir / "training_metadata.json"
predictions_path = run_dir / "validation_predictions.parquet"

joblib.dump(model_x, model_x_path)
joblib.dump(model_y, model_y_path)

trained_feature_columns = list(X_train.columns)

with open(features_path, "w") as f:
    json.dump(trained_feature_columns, f, indent=2)

pred_df.to_parquet(predictions_path)

metadata = {
    "run_name": "final_deploy_xgboost",
    "created_at": datetime.now().isoformat(),
    "input_dataset": str(input_path),
    "n_rows_total": int(df.shape[0]),
    "n_rows_train": int(X_train.shape[0]),
    "n_rows_valid": int(X_valid.shape[0]),
    "target_columns": target_cols,
    "feature_columns_raw": feature_cols,
    "feature_columns_trained": trained_feature_columns,
    "categorical_columns": categorical_cols,
    "metrics_ball_land_x": metrics_x,
    "metrics_ball_land_y": metrics_y,
    "baseline_metrics_ball_land_x": baseline_metrics_x,
    "baseline_metrics_ball_land_y": baseline_metrics_y,
    "model_type": "XGBRegressor",
    "model_params": model_x.get_params(),
    "notes": "Final deploy-safe XGBoost models trained without target-derived leakage features."
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved model_x to:", model_x_path)
print("Saved model_y to:", model_y_path)
print("Saved features to:", features_path)
print("Saved metadata to:", metadata_path)
print("Saved validation predictions to:", predictions_path)

Saved model_x to: C:\Users\jmontanez\Documents\nfl-big-data-bowl-2026\models\final_deploy_xgboost\model_ball_land_x.pkl
Saved model_y to: C:\Users\jmontanez\Documents\nfl-big-data-bowl-2026\models\final_deploy_xgboost\model_ball_land_y.pkl
Saved features to: C:\Users\jmontanez\Documents\nfl-big-data-bowl-2026\models\final_deploy_xgboost\feature_columns.json
Saved metadata to: C:\Users\jmontanez\Documents\nfl-big-data-bowl-2026\models\final_deploy_xgboost\training_metadata.json
Saved validation predictions to: C:\Users\jmontanez\Documents\nfl-big-data-bowl-2026\models\final_deploy_xgboost\validation_predictions.parquet


## Summary

In this notebook, we trained the final deployable XGBoost model using only inference-available features.

This final pipeline:

- removes target-derived leakage,
- preserves the strongest deploy-safe engineered signals,
- produces reproducible model artifacts,
- and serves as the foundation for the Hugging Face application.

The next step is to connect these artifacts to an interactive Gradio interface for end-user prediction.